# PZ data challenge: task set 3 within and across simulations

How well do the task set 3 (spec + COSMOS) models trained on Flagship do on Flagship, and on Cardinal, a simulation they never saw? Four evaluations: Y1 and Y10, each tested on

- **Flagship → Flagship**: the held-out 20% of the Flagship training file (`train_test_split` with `VAL_FRAC` and `SEED`, as in the task-set notebook; the model was trained on the other 80%), and
- **Flagship → Cardinal**: the same 20% split of the Cardinal training file.

It reads the trained ensembles (the subtask 2 `.pkl` model files) that `pz_challenge_taskset3_e2e.ipynb` wrote to `OUT_DIR` and trains nothing; `run_task3_transfer.py` runs both. The p(z)s are predicted here with the current `znn` code (a Gaussian with the mean and std of the predictions), with the same photometric resampling as the submission (`N_SAMPLES` draws), and scored with the challenge metrics and on their stacked n(z).

The held-out predictions are cached in `CACHE_DIR`, shared with `pz_challenge_results.ipynb` (the mock evaluation of all task sets, for submissions). Delete the cache to recompute, e.g. after changing how `znn` builds the p(z). Figures (`.pdf`) and LaTeX tables (`.tex`) are saved to `PLOT_DIR`, prefixed so they never overwrite those of `pz_challenge_results.ipynb`.

Run this from the repo root with the `qp` conda env as the kernel.


## Configuration

In [ ]:
import znn  # run from the repo root so this picks up the local source

# the models: task set TASKSET trained on TRAIN_SIM at each depth, tested on the held-out split of each of TEST_SIMS
TASKSET = 3
TRAIN_SIM = "flagship"
TEST_SIMS = ["flagship", "cardinal"]
SCENARIOS = ["1yr", "10yr"]

DATA_DIR = "../pz_data_challenge/scripts/public"  # as in the task-set notebooks
# Which run to read; must match the task-set notebook: "popcosmos" (the pre-training shipped with znn) or None (from scratch)
PRETRAINING = "popcosmos"
# Method version, appended to OUT_DIR ("" for none): the RUN of the task-set notebook
RUN = "kfold_frombase"  # set by run_task3_transfer.py
OUT_DIR = ("outputs/znn" if PRETRAINING is None else f"outputs/znn_pretrained_{PRETRAINING}") + (f"_{RUN}" if RUN else "")  # the model files written by the task-set notebook
CACHE_DIR = f"{OUT_DIR}_predictions"  # held-out predictions, shared with pz_challenge_results.ipynb
PLOT_DIR = "plots"  # figures and LaTeX tables are saved here, prefixed with the run name and the evaluation

# the held-out split: must match the task-set notebook
VAL_FRAC = 0.2
SEED = 42
# prediction: photometric noise draws per object (converged by 50 on Cardinal Y1) and their seed
N_SAMPLES = 50
NOISE_SEED = 42


In [ ]:
import itertools
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qp
from qp.metrics.pit import PIT
from sklearn.model_selection import train_test_split

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# one colour per task set, as in pz_challenge_results.ipynb; its colour map is used for 2D densities
TASKSET_CMAPS = {1: "Blues", 2: "Greens", 3: "Oranges", 4: "Purples"}
COLORS = {t: plt.get_cmap(cmap)(0.7) for t, cmap in TASKSET_CMAPS.items()}
TRAIN_COLOR = "0.6"
# how sims and depths are written in plots and tables
SIM_LABELS = {"cardinal": "Cardinal", "flagship": "Flagship"}
DEPTH_LABELS = {"1yr": "Y1", "10yr": "Y10"}
INDEX_LABELS = {"test": "Tested on", "scenario": "Depth"}
# prefix of the saved figures and tables
PREFIX = f"{os.path.basename(OUT_DIR)}_taskset{TASKSET}_{TRAIN_SIM}_transfer"


def transfer_label(test_sim):
    # training -> test simulation
    return f"{SIM_LABELS[TRAIN_SIM]} → {SIM_LABELS[test_sim]}"


def save(fig, name):
    fig.savefig(os.path.join(PLOT_DIR, f"{PREFIX}_{name}.pdf"), bbox_inches="tight")


def save_latex(df, name, columns, caption):
    # Write df as a booktabs LaTeX table (columns: {df column: LaTeX header}) to PLOT_DIR and print it. Values of
    # 10 or more (the Anderson-Darling statistic) get one decimal, the rest four.
    df = df[list(columns)].rename(columns=columns, index=SIM_LABELS | DEPTH_LABELS)
    tex = df.rename_axis([INDEX_LABELS[n] for n in df.index.names]).to_latex(
        float_format=lambda v: f"{v:.1f}" if abs(v) >= 10 else f"{v:.4f}", multirow=True, caption=caption,
        label=f"tab:{name}", position="htbp",
    )
    # a rule between groups of the first index level only
    tex = re.sub(r"^(?:\\cline\{\d+-\d+\} ?)+\n", lambda m: "\\midrule\n" if "\\cline{1-" in m[0] else "", tex, flags=re.M)
    tex = tex.replace("\\midrule\n\\bottomrule", "\\bottomrule")
    with open(os.path.join(PLOT_DIR, f"{PREFIX}_{name}.tex"), "w") as f:
        f.write(tex)
    print(tex)


def path(kind, taskset, sim, scenario):
    if kind in ("training", "test"):
        return os.path.join(DATA_DIR, f"pz_challenge_taskset_{taskset}_{sim}_{kind}_{scenario}.hdf5")
    return os.path.join(OUT_DIR, f"pz_challenge_taskset_{taskset}_{sim}_{kind}_{scenario}.pkl")


missing = [p for s in SCENARIOS if not os.path.exists(p := path("pz_model", TASKSET, TRAIN_SIM, s))]
assert not missing, f"missing model files: {missing}"


## Reference redshifts

These are the same labels as in the task-set notebooks: `redshift` for spectroscopically selected objects and `redshift_manyband` for COSMOS-only objects, so the scores include the label noise of the COSMOS-only objects, on both simulations.


In [ ]:
SPEC_SURVEYS = ["DEEP2_LSST", "DESI_BGS", "DESI_ELG_LOP", "DESI_LRG", "VVDSf02", "zCOSMOS"]


def reference_redshifts(train):
    # Reference redshift per training object.
    z_spec = train["redshift"].to_numpy()
    if "COSMOS" not in train:  # task sets 1 and 2
        return z_spec
    is_spec = train[[c for c in SPEC_SURVEYS if c in train]].astype(bool).any(axis=1).to_numpy()
    is_cosmos_only = train["COSMOS"].astype(bool).to_numpy() & ~is_spec
    z_ref = np.where(is_cosmos_only, train["redshift_manyband"].to_numpy(), z_spec)
    assert np.isfinite(z_ref).all()
    return z_ref


# Held-out scores

Each model is run on the held-out split of Flagship and of Cardinal. Cardinal has the same task set 3 selection and bands as Flagship but a different simulated galaxy population, so Flagship → Cardinal measures how well the model carries over to galaxies unlike the ones it was trained on. Cardinal's test file has no redshifts, so its training file is used; its held-out split is the same 20k objects a Cardinal-trained model is scored on in the task-set notebook.


In [ ]:
def heldout_predictions(taskset, sim, scenario, test_sim=None):
    # The model trained on sim, run on the held-out split of test_sim's training file (default: sim's own). Returns the
    # held-out reference z, i-mag, zmode, the p(z) mean and std, and the ensemble's p(z) (a qp ensemble) under "pz".
    # Cached in CACHE_DIR: the arrays in <stem>.npz, the p(z) in <stem>.hdf5.
    test_sim = test_sim or sim
    on = "" if test_sim == sim else f"_on_{test_sim}"
    stem = os.path.join(CACHE_DIR, f"taskset_{taskset}_{sim}{on}_{scenario}_pz")
    if os.path.exists(f"{stem}.npz") and os.path.exists(f"{stem}.hdf5"):
        return dict(np.load(f"{stem}.npz")) | {"pz": qp.read(f"{stem}.hdf5")}

    train = znn.read_catalog(path("training", taskset, test_sim, scenario))
    z_ref = reference_redshifts(train)
    _, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
    val = train.iloc[idx_val]

    trained_models, features = znn.load_ensemble_file(path("pz_model", taskset, sim, scenario))
    errors = {col: f"{col}_err" for col, _ in features["bands"].values()}
    pz = znn.predict_from_catalog(
        trained_models, val, **features, errors=errors, n_samples=N_SAMPLES, rng=np.random.default_rng(NOISE_SEED)
    )
    out = {
        "z_ref": z_ref[idx_val],
        "mag_i": val["mag_i_lsst"].to_numpy(),
        "zmode": np.squeeze(pz.ancil["zmode"]),
        "mean": np.squeeze(pz.mean()),
        "std": np.squeeze(pz.std()),
    }
    np.savez(f"{stem}.npz", **out)
    pz.write_to(f"{stem}.hdf5")
    return out | {"pz": pz}


# the evaluations, keyed by (test sim, depth)
EVALS = list(itertools.product(TEST_SIMS, SCENARIOS))
heldout = {}
for test_sim, scenario in EVALS:
    print("held-out predictions:", transfer_label(test_sim), DEPTH_LABELS[scenario])
    heldout[(test_sim, scenario)] = heldout_predictions(TASKSET, TRAIN_SIM, scenario, test_sim=test_sim)


In [ ]:
def challenge_metrics(z_ref, zmode, pz):
    # The challenge's seven scored metrics, computed as in the task-set notebooks.
    dz = (zmode - z_ref) / (1 + z_ref)
    bias, _, scatter, _, abs_outlier_rate = znn.get_biweight_mean_sigma_outlier(dz)
    pit = PIT(pz, z_ref)
    return {
        "mean": bias,
        "std": scatter,
        "abs_outlier_rate": abs_outlier_rate,
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        # qp derives this from a spline fit to the PIT quantiles and it breaks down (negative or huge values)
        # when PIT piles up at 0 or 1; the challenge evaluator uses the same call, so it is what gets scored
        "outlier": pit.evaluate_PIT_outlier_rate(),
    }, pit.pit_samps


results, pits = {}, {}
for c, h in heldout.items():
    results[c], pits[c] = challenge_metrics(h["z_ref"], h["zmode"], h["pz"])

# ks is shown on the PIT histograms; ks and CvM are graded in pz_challenge_results.ipynb
table = pd.DataFrame(results).T[["mean", "std", "abs_outlier_rate", "ksamp", "outlier"]]
table.index.names = ["test", "scenario"]
table.round(4)

### p(z) metrics

What each column of the table above measures, all on the held-out set. Lower is better for all of them except `mean`, which should be close to 0.

Point-estimate metrics, on Δz = (zmode − z_ref)/(1 + z_ref). They are computed after 3σ-clipping Δz four times, as in `znn.get_biweight_mean_sigma_outlier`:
- `mean`: the bias, the biweight location of the clipped Δz.
- `std`: the scatter, the biweight scale of the clipped Δz (a robust standard deviation).
- `abs_outlier_rate`: the fraction of *all* objects (not clipped) with |Δz| > 0.2.

Calibration metrics, on the PIT (the CDF of each p(z) evaluated at its z_ref). These should be uniform on [0, 1] if the p(z)s are calibrated:
- `ksamp`: Anderson–Darling k-sample statistic between the PIT values and a uniform sample. It weights the tails (PIT near 0 and 1) more than a KS test does, so it reacts most to catastrophic misses. It grows with the number of objects.
- `outlier`: the fraction of PIT values below 0.0001 or above 0.9999, i.e. z_ref far out in the tails of its p(z). qp estimates it from a spline fit to the PIT quantiles, which breaks down (negative or huge values) when PIT piles up at 0 or 1. The challenge evaluator uses the same call, so this is the value that gets scored.

The challenge also scores the KS and Cramér–von Mises statistics of the PIT. They are left out of the table; the PIT histograms below show the calibration directly.


In [ ]:
save_latex(
    table,
    "pz_metrics",
    {
        "mean": r"bias",
        "std": r"$\sigma$",
        "abs_outlier_rate": r"$f_{|\Delta z| > 0.2}$",
        "ksamp": r"AD",
        "outlier": r"$f_\mathrm{PIT\,out}$",
    },
    r"p(z) metrics of the task set 3 models trained on Flagship, on the held-out split of Flagship and of Cardinal. Point estimates: bias, scatter $\sigma$ and outlier fraction of "
    r"$\Delta z = (z_\mathrm{mode} - z_\mathrm{ref})/(1 + z_\mathrm{ref})$. Calibration: Anderson--Darling "
    r"statistic of the PIT against uniform and the fraction of PIT outliers.",
)


### PIT

PIT histograms on the held-out set. A flat histogram means the p(z) is calibrated. A U shape means it is too narrow, and a hump means it is too wide.

In [ ]:
from matplotlib.ticker import MaxNLocator

pit_bins = np.linspace(0, 1, 26)
top = 0
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True, gridspec_kw=dict(hspace=0, wspace=0))
for row, test_sim in enumerate(TEST_SIMS):
    for col, scenario in enumerate(SCENARIOS):
        ax = axes[row, col]
        heights, _, _ = ax.hist(pits[(test_sim, scenario)], pit_bins, density=True, color=COLORS[TASKSET],
                histtype="stepfilled", alpha=0.8, ec="white", lw=0.5)
        ax.axhline(1, color="k", lw=0.8, ls=":")
        ks = results[(test_sim, scenario)]["ks"]
        ax.text(0.5, 0.95, f"KS {ks:.2f}", transform=ax.transAxes, ha="center", va="top", fontsize=10,
                bbox=dict(fc="white", ec="none", alpha=0.8, pad=1))
        top = max(top, heights.max())
        if row == 0:
            ax.set_title(DEPTH_LABELS[scenario], fontsize=12)
        if col == 0:
            ax.set_ylabel(transfer_label(test_sim))
        if row == len(TEST_SIMS) - 1:
            ax.set_xlabel("PIT")
# with no gaps, keep the ticks off the panel edges so labels of neighbouring panels do not touch
axes[0, 0].set_xlim(0, 1)
axes[0, 0].set_ylim(0, 1.25 * top)  # headroom for the KS labels
axes[0, 0].set_xticks([0, 0.25, 0.5, 0.75])
axes[0, 0].yaxis.set_major_locator(MaxNLocator(nbins=4, prune="upper"))
save(fig, "pit")
plt.show()


### Point estimates

`zmode` against the reference redshift on the held-out set (log-density 2D histogram). The dashed lines mark |Δz|/(1+z) = 0.2, the outlier threshold.

In [ ]:
from matplotlib.colors import LogNorm

zb = np.linspace(0, 3, 121)
fig, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True, gridspec_kw=dict(hspace=0, wspace=0))
for row, test_sim in enumerate(TEST_SIMS):
    for col, scenario in enumerate(SCENARIOS):
        ax = axes[row, col]
        h = heldout[(test_sim, scenario)]
        ax.hist2d(h["z_ref"], h["zmode"], bins=[zb, zb], cmap=TASKSET_CMAPS[TASKSET], norm=LogNorm(), rasterized=True)
        ax.plot(zb, zb, color="k", lw=0.6)
        for sgn in (-1, 1):
            ax.plot(zb, zb + sgn * 0.2 * (1 + zb), color="k", lw=0.6, ls="--")
        v = table.loc[(test_sim, scenario)]
        ax.text(0.04, 0.96, f"$\\delta_z$ {v['mean']:+.4f}\nσ {v['std']:.4f}\nout {v['abs_outlier_rate']:.3f}",
                transform=ax.transAxes, va="top", fontsize=9)
        if row == 0:
            ax.set_title(DEPTH_LABELS[scenario], fontsize=12)
        if col == 0:
            ax.set_ylabel(f"{transfer_label(test_sim)}\n$\\hat{{z}}$")
        if row == len(TEST_SIMS) - 1:
            ax.set_xlabel("$z$")
axes[0, 0].set_xlim(0, 3)
axes[0, 0].set_ylim(0, 3)
# ticks stop short of 3 so labels of neighbouring panels do not touch
axes[0, 0].set_xticks(np.arange(0, 3, 0.5))
axes[0, 0].set_yticks(np.arange(0, 3, 0.5))
save(fig, "zphot_vs_zref")
plt.show()


### n(z)

The predicted n(z) of each held-out set against its reference redshifts. One redshift is drawn from every object's p(z) and the draws are histogrammed. Repeating this `N_NZ_DRAWS` times gives the predicted n(z) in each bin (points: the mean over the draws, i.e. the stacked p(z)) and its uncertainty (error bars: the standard deviation over the draws). If the p(z)s are calibrated, the histogram of the reference redshifts (black) is one more such draw and should agree with the points within the error bars. The grey shadow is the n(z) of the Flagship training split the model was trained on: on Flagship it matches the held-out histogram, on Cardinal it shows how far Cardinal's redshift distribution is from what the model learned.

The table compares the two distributions, predicted minus reference: `Δ<z>` is the change in mean redshift and `Δσ_z` the change in width (standard deviation), both averaged over the draws. `KS` is the two-sample Kolmogorov-Smirnov statistic between the reference redshifts and all the draws pooled, i.e. the stacked p(z).


In [ ]:
from matplotlib.ticker import MaxNLocator
from scipy.stats import ks_2samp

NZ_BINS = np.linspace(0, 3, 61)
NZ_CENTRES = 0.5 * (NZ_BINS[1:] + NZ_BINS[:-1])
N_NZ_DRAWS = 1_000
nz_rng = np.random.default_rng(NOISE_SEED)


def nz_hist(z):
    # n(z) normalised over all objects, so draws outside NZ_BINS (e.g. z < 0) lower it rather than being dropped
    return np.histogram(z, NZ_BINS)[0] / (len(z) * np.diff(NZ_BINS))


# the 80% of the training file each model was trained on, per depth
nz_train = {}
for scenario in SCENARIOS:
    train = znn.read_catalog(path("training", TASKSET, TRAIN_SIM, scenario))
    idx_train, _ = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
    nz_train[scenario] = nz_hist(reference_redshifts(train)[idx_train])

nz_pred, nz_true, nz_metrics = {}, {}, {}
for c, h in heldout.items():
    z_draws = h["pz"].rvs(size=N_NZ_DRAWS, random_state=nz_rng)  # (objects, draws)
    nz_pred[c] = np.stack([nz_hist(zd) for zd in z_draws.T])
    nz_true[c] = nz_hist(h["z_ref"])
    nz_metrics[c] = {
        "Δ<z>": z_draws.mean(axis=0).mean() - h["z_ref"].mean(),
        "Δσ_z": z_draws.std(axis=0).mean() - h["z_ref"].std(),
        "KS": ks_2samp(h["z_ref"], z_draws.ravel()).statistic,
    }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True, gridspec_kw=dict(hspace=0, wspace=0))
for row, test_sim in enumerate(TEST_SIMS):
    for col, scenario in enumerate(SCENARIOS):
        ax = axes[row, col]
        c = (test_sim, scenario)
        ax.stairs(nz_train[scenario], NZ_BINS, fill=True, color=TRAIN_COLOR, alpha=0.35, lw=0, label="Training")
        ax.stairs(nz_true[c], NZ_BINS, color="k", lw=1.0, label="Truth")
        ax.errorbar(NZ_CENTRES, nz_pred[c].mean(axis=0), nz_pred[c].std(axis=0), fmt=".", ms=3, lw=1, capsize=0,
                    color=COLORS[TASKSET], label="Predicted")
        m = nz_metrics[c]
        #ax.text(0.97, 0.92, f"Δ<z> {m['Δ<z>']:+.4f}\nΔσ_z {m['Δσ_z']:+.4f}\nKS {m['KS']:.3f}",
        #        transform=ax.transAxes, ha="right", va="top", fontsize=9)
        if row == 0:
            ax.set_title(DEPTH_LABELS[scenario], fontsize=12)
        if col == 0:
            ax.set_ylabel(f"{transfer_label(test_sim)}\nn(z)")
        if row == len(TEST_SIMS) - 1:
            ax.set_xlabel("z")
axes[0, 0].legend(fontsize=9, frameon=False, loc="center right")
# with no gaps, keep the ticks off the panel edges so labels of neighbouring panels do not touch
axes[0, 0].set_xlim(0, 3)
axes[0, 0].set_xticks(np.arange(0, 3, 0.5))
axes[0, 0].yaxis.set_major_locator(MaxNLocator(nbins=4, prune="upper"))
save(fig, "nz_heldout")
plt.show()

nz_table = pd.DataFrame.from_dict(nz_metrics, orient="index")
nz_table.index.names = table.index.names
save_latex(
    nz_table,
    "nz_metrics",
    {"Δ<z>": r"$\Delta\langle z \rangle$", "Δσ_z": r"$\Delta\sigma_z$", "KS": r"KS"},
    r"Predicted against held-out n(z) of the task set 3 models trained on Flagship, on Flagship and on Cardinal: change in mean redshift $\Delta\langle z \rangle$ and in width $\Delta\sigma_z$ "
    r"(predicted minus reference), and the two-sample Kolmogorov--Smirnov statistic between the reference redshifts and "
    r"redshifts drawn from the p(z)s.",
)
nz_table.round(4)
